# Wyckoff Multiset Coverage

Compare train/model Wyckoff-letter multiset coverage against theoretically possible multisets with up to `MAX_ATOMS` atoms per unit cell.

Euclidean-normalizer relabelings are handled with `pymatgen.analysis.prototypes.WYCKOFF_POSITION_RELAB_DICT`, matching `src.wyckoff_match.WyckoffData.letter_key`.

Theoretical counts enforce per-Wyckoff-letter occupancy limits: fixed positions can be occupied once, and positions with free parameters can repeat up to the atom-count limit.


In [ ]:
from __future__ import annotations

import gzip
import hashlib
import json
import pickle
import sys
from collections import Counter, defaultdict
from importlib import resources
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymatgen.analysis.prototypes as prototypes
from pymatgen.analysis.prototypes import (
    WYCKOFF_MULTIPLICITY_DICT,
    WYCKOFF_POSITION_RELAB_DICT,
)

NOTEBOOK_DIR = (
    Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from notebook_constants import (  # noqa: E402
    CRYSTAL_SYSTEM_ORDER,
    EHULL_RELAXED_FILE,
    EHULL_RELAXED_INFO_FILE,
    METASTABLE_EHULL_MAX,
    SMACT_VALIDITY_FILE,
)
from notebook_utils import (  # noqa: E402
    crystal_system_from_spg_num,
    find_repo_root,
    load_pickle_gz,
    load_relax_convergence,
    load_relaxed_ehulls,
    load_smact_validity,
)

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.config import INPUT_DIR, RESULTS_DIR  # noqa: E402

NOTEBOOK_DIR = REPO_ROOT / "notebooks"
WYCKOFF_REPR_FILE = "wyckoff_repr_s=0.01.pkl.gz"
MAX_ATOMS = 20
OCCUPANCY_MODE = "bounded"
CACHE_DIR = NOTEBOOK_DIR / ".cache" / "enumerate_wyckoff"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

WYCKOFF_PARAMS_PATH = resources.files(prototypes).joinpath(
    "wyckoff-position-params.json.gz"
)
with gzip.open(WYCKOFF_PARAMS_PATH, "rt") as file:
    WYCKOFF_POSITION_PARAMS = json.load(file)
WYCKOFF_PARAMS_DIGEST = hashlib.sha256(
    json.dumps(WYCKOFF_POSITION_PARAMS, sort_keys=True).encode()
).hexdigest()[:16]

MODEL_PATHS = {
    path.parent.name: path
    for path in sorted(RESULTS_DIR.glob(f"*/{WYCKOFF_REPR_FILE}"))
    if path.parent.name != "train"
}
TRAIN_PATH = RESULTS_DIR / "train" / WYCKOFF_REPR_FILE
SUBSET_LABELS = {
    "all": "all generated structures",
    "metastable_smact_valid": "metastable and SMACT-valid structures",
}

print(f"Repository: {REPO_ROOT}")
print(f"Cache: {CACHE_DIR}")
print(f"Train: {TRAIN_PATH}")
print(f"Models: {', '.join(MODEL_PATHS) if MODEL_PATHS else '(none found)'}")

In [ ]:
CATEGORY_COLORS = {
    "both": "#2a9d8f",
    "train_only": "#e9c46a",
    "model_only": "#e76f51",
    "neither": "#d0d0d0",
}
CATEGORY_LABELS = {
    "both": "both train and model",
    "train_only": "train only",
    "model_only": "model only",
    "neither": "neither",
}


def cache_key(parts: dict[str, Any]) -> str:
    payload = json.dumps(parts, sort_keys=True, default=str).encode()
    return hashlib.sha256(payload).hexdigest()[:16]


def load_cache(path: Path) -> Any | None:
    if not path.exists():
        return None
    with gzip.open(path, "rb") as file:
        return pickle.load(file)  # noqa: S301


def save_cache(path: Path, value: Any) -> Any:
    path.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(path, "wb") as file:
        pickle.dump(value, file)
    return value


def cached(path: Path, builder):
    value = load_cache(path)
    if value is not None:
        print(f"Loaded cache: {path.name}")
        return value
    value = builder()
    print(f"Writing cache: {path.name}")
    return save_cache(path, value)

In [ ]:
def relabelings_for_spg(spg_num: int) -> list[dict[int, str]]:
    return WYCKOFF_POSITION_RELAB_DICT.get(str(spg_num), []) or [
        {ord(letter): letter for letter in WYCKOFF_MULTIPLICITY_DICT[str(spg_num)]}
    ]


def relabeling_cycles(spg_num: int, trans: dict[int, str]) -> list[tuple[str, ...]]:
    multiplicities = WYCKOFF_MULTIPLICITY_DICT[str(spg_num)]
    seen: set[str] = set()
    cycles: list[tuple[str, ...]] = []
    for letter in sorted(multiplicities):
        if letter in seen:
            continue
        cycle = []
        current = letter
        while current not in seen:
            if current not in multiplicities:
                raise ValueError(
                    f"SG {spg_num} relabeling maps to unknown letter {current!r}"
                )
            seen.add(current)
            cycle.append(current)
            current = current.translate(trans)
        cycles.append(tuple(cycle))
    return cycles


def letter_max_counts(spg_num: int, max_atoms: int = MAX_ATOMS) -> dict[str, int]:
    multiplicities = WYCKOFF_MULTIPLICITY_DICT[str(spg_num)]
    position_params = WYCKOFF_POSITION_PARAMS[str(spg_num)]
    max_counts = {}
    for letter, multiplicity in multiplicities.items():
        n_free = int(position_params.get(letter, 0))
        max_counts[letter] = max_atoms // int(multiplicity) if n_free > 0 else 1
    return max_counts


def count_bounded_weighted_solutions_leq(
    weights: list[int], bounds: list[int], max_atoms: int
) -> int:
    dp = [0] * (max_atoms + 1)
    dp[0] = 1
    for weight, bound in zip(weights, bounds, strict=True):
        next_dp = [0] * (max_atoms + 1)
        for total, ways in enumerate(dp):
            if ways == 0:
                continue
            for count in range(bound + 1):
                next_total = total + count * weight
                if next_total > max_atoms:
                    break
                next_dp[next_total] += ways
        dp = next_dp
    return sum(dp[1:])


def theoretical_count_for_spg(spg_num: int, max_atoms: int = MAX_ATOMS) -> int:
    multiplicities = WYCKOFF_MULTIPLICITY_DICT[str(spg_num)]
    max_counts = letter_max_counts(spg_num, max_atoms)
    relabelings = relabelings_for_spg(spg_num)
    fixed_total = 0
    for trans in relabelings:
        cycles = relabeling_cycles(spg_num, trans)
        cycle_weights = [
            sum(int(multiplicities[letter]) for letter in cycle) for cycle in cycles
        ]
        cycle_bounds = [min(max_counts[letter] for letter in cycle) for cycle in cycles]
        fixed_total += count_bounded_weighted_solutions_leq(
            cycle_weights, cycle_bounds, max_atoms
        )
    if fixed_total % len(relabelings) != 0:
        raise ValueError(f"SG {spg_num} Burnside count is not integral")
    return fixed_total // len(relabelings)


def build_theoretical_counts() -> pd.DataFrame:
    rows = []
    for spg_num in range(1, 231):
        rows.append(
            {
                "spg_num": spg_num,
                "crystal_system": crystal_system_from_spg_num(spg_num),
                "theoretical": theoretical_count_for_spg(spg_num, MAX_ATOMS),
                "n_relabelings": len(relabelings_for_spg(spg_num)),
            }
        )
    return pd.DataFrame(rows)


theory_cache = CACHE_DIR / (
    f"theoretical_counts_occupancy={OCCUPANCY_MODE}_max_atoms={MAX_ATOMS}_"
    f"params={WYCKOFF_PARAMS_DIGEST}.pkl.gz"
)
theory = cached(theory_cache, build_theoretical_counts)
display(theory.head())
display(theory.sort_values("theoretical", ascending=False).head(15))
print(f"Total theoretical canonical multisets: {theory['theoretical'].sum():,}")

In [ ]:
def key_atom_count(spg_num: int, key: tuple[str, ...]) -> int:
    multiplicities = WYCKOFF_MULTIPLICITY_DICT[str(spg_num)]
    return sum(int(multiplicities[letter]) for letter in key)


def key_occupancy_violations(
    spg_num: int, key: tuple[str, ...], max_atoms: int = MAX_ATOMS
) -> list[str]:
    counts = Counter(key)
    max_counts = letter_max_counts(spg_num, max_atoms)
    return sorted(
        letter for letter, count in counts.items() if count > max_counts[letter]
    )


def canonical_observed_key(data) -> tuple[int, tuple[str, ...]]:
    if not data.letter_key:
        raise ValueError("WyckoffData has no letter_key entries")
    spg_num = int(data.spg_num)
    key = min(tuple(letter_key) for letter_key in data.letter_key)
    return spg_num, key


def filter_digest(include_indices: set[int] | None) -> str:
    if include_indices is None:
        return "all"
    payload = ",".join(str(idx) for idx in sorted(include_indices)).encode()
    return hashlib.sha256(payload).hexdigest()[:16]


def observed_sets_by_spg(
    path: Path,
    max_atoms: int = MAX_ATOMS,
    *,
    include_indices: set[int] | None = None,
) -> dict[int, set[tuple[str, ...]]]:
    data = load_pickle_gz(path)
    by_spg: dict[int, set[tuple[str, ...]]] = defaultdict(set)
    dropped_atom_count = 0
    dropped_occupancy = 0
    skipped_by_filter = 0
    for idx, record in enumerate(data):
        if include_indices is not None and idx not in include_indices:
            skipped_by_filter += 1
            continue
        spg_num, key = canonical_observed_key(record)
        multiplicities = WYCKOFF_MULTIPLICITY_DICT[str(spg_num)]
        invalid_letters = sorted(set(key) - set(multiplicities))
        if invalid_letters:
            raise ValueError(
                f"{path} has invalid SG {spg_num} letters {invalid_letters}"
            )
        if key_atom_count(spg_num, key) > max_atoms:
            dropped_atom_count += 1
            continue
        if key_occupancy_violations(spg_num, key, max_atoms):
            dropped_occupancy += 1
            continue
        by_spg[spg_num].add(key)
    print(
        f"{path.parent.name}: {len(data):,} records, "
        f"{sum(len(v) for v in by_spg.values()):,} unique keys <= {max_atoms} atoms, "
        f"{skipped_by_filter:,} skipped by subset filter, "
        f"{dropped_atom_count:,} dropped by atom count, "
        f"{dropped_occupancy:,} dropped by occupancy limits"
    )
    return dict(by_spg)


def observed_cache_path(
    label: str,
    path: Path,
    subset: str,
    include_indices: set[int] | None = None,
) -> Path:
    stat = path.stat()
    digest = cache_key(
        {
            "label": label,
            "path": path,
            "mtime_ns": stat.st_mtime_ns,
            "size": stat.st_size,
            "max_atoms": MAX_ATOMS,
            "occupancy_mode": OCCUPANCY_MODE,
            "wyckoff_params_digest": WYCKOFF_PARAMS_DIGEST,
            "subset": subset,
            "filter_count": None if include_indices is None else len(include_indices),
            "filter_digest": filter_digest(include_indices),
        }
    )
    return CACHE_DIR / f"observed_{label}_{subset}_{digest}.pkl.gz"


def cached_observed_sets(
    label: str,
    path: Path,
    subset: str,
    include_indices: set[int] | None = None,
) -> dict[int, set[tuple[str, ...]]]:
    cache_path = observed_cache_path(label, path, subset, include_indices)
    observed_cache_paths[(subset, label)] = cache_path
    return cached(
        cache_path,
        lambda: observed_sets_by_spg(path, include_indices=include_indices),
    )


def metastable_smact_valid_indices(model: str, expected: int) -> set[int]:
    gen_dir = INPUT_DIR / "gen" / "preprocessed" / model
    ehull = load_relaxed_ehulls(gen_dir / EHULL_RELAXED_FILE, expected)
    converged = load_relax_convergence(gen_dir / EHULL_RELAXED_INFO_FILE, expected)
    smact_valid = load_smact_validity(gen_dir / SMACT_VALIDITY_FILE, expected)
    mask = converged & (ehull <= METASTABLE_EHULL_MAX) & smact_valid
    return set(int(idx) for idx in np.flatnonzero(mask))


observed_cache_paths: dict[tuple[str, str], Path] = {}
train_sets = cached_observed_sets("train", TRAIN_PATH, "all")
model_sets_all = {
    model: cached_observed_sets(model, path, "all")
    for model, path in MODEL_PATHS.items()
}
model_subset_indices = {
    model: metastable_smact_valid_indices(model, len(load_pickle_gz(path)))
    for model, path in MODEL_PATHS.items()
}
model_sets_metastable_smact_valid = {
    model: cached_observed_sets(
        model,
        path,
        "metastable_smact_valid",
        model_subset_indices[model],
    )
    for model, path in MODEL_PATHS.items()
}
model_sets_by_subset = {
    "all": model_sets_all,
    "metastable_smact_valid": model_sets_metastable_smact_valid,
}

In [ ]:
def observed_sets_to_frame(
    label: str, subset: str, by_spg: dict[int, set[tuple[str, ...]]]
) -> pd.DataFrame:
    rows = []
    for spg_num, keys in sorted(by_spg.items()):
        for key in sorted(keys):
            rows.append(
                {
                    "dataset": label,
                    "subset": subset,
                    "spg_num": spg_num,
                    "crystal_system": crystal_system_from_spg_num(spg_num),
                    "wyckoff_multiset": " ".join(key),
                    "atom_count": key_atom_count(spg_num, key),
                }
            )
    return pd.DataFrame(rows)


observed_multisets = pd.concat(
    [observed_sets_to_frame("train", "all", train_sets)]
    + [
        observed_sets_to_frame(model, subset, sets)
        for subset, model_sets in model_sets_by_subset.items()
        for model, sets in model_sets.items()
    ],
    ignore_index=True,
)
observed_multisets.to_csv(
    CACHE_DIR
    / f"observed_multisets_occupancy={OCCUPANCY_MODE}_max_atoms={MAX_ATOMS}.csv",
    index=False,
)
display(observed_multisets.head())
display(
    observed_multisets.groupby(["subset", "dataset"]).size().rename("unique_multisets")
)


def coverage_table_for_model(
    subset: str, model: str, model_by_spg: dict[int, set[tuple[str, ...]]]
) -> pd.DataFrame:
    rows = []
    theory_by_spg = theory.set_index("spg_num")["theoretical"].to_dict()
    for spg_num in range(1, 231):
        train = train_sets.get(spg_num, set())
        model_set = model_by_spg.get(spg_num, set())
        both = train & model_set
        train_only = train - model_set
        model_only = model_set - train
        union = train | model_set
        theoretical = int(theory_by_spg[spg_num])
        neither = theoretical - len(union)
        if neither < 0:
            raise ValueError(
                f"Observed union for SG {spg_num} exceeds theoretical count: "
                f"{len(union)} > {theoretical}"
            )
        rows.append(
            {
                "subset": subset,
                "model": model,
                "spg_num": spg_num,
                "crystal_system": crystal_system_from_spg_num(spg_num),
                "theoretical": theoretical,
                "both": len(both),
                "train_only": len(train_only),
                "model_only": len(model_only),
                "neither": neither,
                "observed_union": len(union),
            }
        )
    frame = pd.DataFrame(rows)
    assert (
        frame[["both", "train_only", "model_only", "neither"]].sum(axis=1)
        == frame["theoretical"]
    ).all()
    assert (
        frame[["both", "train_only", "model_only"]].sum(axis=1)
        == frame["observed_union"]
    ).all()
    return frame


coverage_digest = cache_key(
    {
        "max_atoms": MAX_ATOMS,
        "occupancy_mode": OCCUPANCY_MODE,
        "wyckoff_params_digest": WYCKOFF_PARAMS_DIGEST,
        "theory_cache": theory_cache.name,
        "observed_caches": {
            f"{subset}:{label}": cache_path.name
            for (subset, label), cache_path in observed_cache_paths.items()
        },
    }
)
coverage_cache = CACHE_DIR / f"coverage_{coverage_digest}.pkl.gz"
coverage = cached(
    coverage_cache,
    lambda: pd.concat(
        [
            coverage_table_for_model(subset, model, sets)
            for subset, model_sets in model_sets_by_subset.items()
            for model, sets in model_sets.items()
        ],
        ignore_index=True,
    ),
)
display(coverage.head())
display(
    coverage.groupby(["subset", "model"])[
        ["both", "train_only", "model_only", "observed_union"]
    ]
    .sum()
    .sort_index()
)

COUNT_COLUMNS = [
    "theoretical",
    "both",
    "train_only",
    "model_only",
    "neither",
    "observed_union",
]

coverage_by_crystal_system = (
    coverage.groupby(["subset", "model", "crystal_system"], as_index=False)[
        COUNT_COLUMNS
    ]
    .sum()
    .assign(
        crystal_system=lambda df: pd.Categorical(
            df["crystal_system"], categories=CRYSTAL_SYSTEM_ORDER, ordered=True
        )
    )
    .sort_values(["subset", "model", "crystal_system"])
    .reset_index(drop=True)
)
assert (
    coverage_by_crystal_system[["both", "train_only", "model_only", "neither"]].sum(
        axis=1
    )
    == coverage_by_crystal_system["theoretical"]
).all()
assert (
    coverage_by_crystal_system[["both", "train_only", "model_only"]].sum(axis=1)
    == coverage_by_crystal_system["observed_union"]
).all()
display(coverage_by_crystal_system.head(14))

In [ ]:
def ratio_frame(
    frame: pd.DataFrame, x_column: str, columns: list[str], denominator: str
) -> pd.DataFrame:
    ratios = frame[[x_column, *columns, denominator]].copy()
    for column in columns:
        ratios[column] = ratios[column] / ratios[denominator].where(
            ratios[denominator] != 0, 1
        )
    return ratios


def plot_stacked_ratios(
    frame: pd.DataFrame,
    *,
    model: str,
    columns: list[str],
    denominator: str,
    x_column: str,
    x_label: str,
    title: str,
    figsize: tuple[float, float] = (24, 5),
    tick_labelsize: int = 6,
) -> plt.Axes:
    model_frame = frame.query("model == @model").sort_values(x_column)
    ratios = ratio_frame(model_frame, x_column, columns, denominator)
    ax = ratios.set_index(x_column)[columns].plot(
        kind="bar",
        stacked=True,
        figsize=figsize,
        width=1.0,
        color=[CATEGORY_COLORS[column] for column in columns],
        edgecolor="none",
    )
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel("Ratio")
    ax.set_ylim(0, 1)
    ax.legend(
        [CATEGORY_LABELS[column] for column in columns],
        ncol=len(columns),
        loc="upper center",
        bbox_to_anchor=(0.5, 1.18),
    )
    ax.tick_params(axis="x", labelsize=tick_labelsize)
    return ax


THEORETICAL_COLUMNS = ["both", "train_only", "model_only", "neither"]

for subset, subset_label in SUBSET_LABELS.items():
    subset_coverage = coverage[coverage["subset"] == subset]
    subset_coverage_by_crystal_system = coverage_by_crystal_system[
        coverage_by_crystal_system["subset"] == subset
    ]
    for model in sorted(model_sets_by_subset[subset]):
        plot_stacked_ratios(
            subset_coverage,
            model=model,
            columns=THEORETICAL_COLUMNS,
            denominator="theoretical",
            x_column="spg_num",
            x_label="Space group number",
            title=(
                f"{model}: {subset_label} coverage relative to theoretical "
                f"Wyckoff multisets <= {MAX_ATOMS} atoms"
            ),
        )
        plt.show()

        plot_stacked_ratios(
            subset_coverage_by_crystal_system,
            model=model,
            columns=THEORETICAL_COLUMNS,
            denominator="theoretical",
            x_column="crystal_system",
            x_label="Crystal system",
            title=(
                f"{model}: {subset_label} coverage by crystal system relative to "
                f"theoretical Wyckoff multisets <= {MAX_ATOMS} atoms"
            ),
            figsize=(10, 5),
            tick_labelsize=10,
        )
        plt.show()

In [ ]:
summary = (
    coverage.groupby(["subset", "model"])[
        ["theoretical", "both", "train_only", "model_only", "neither", "observed_union"]
    ]
    .sum()
    .assign(
        theoretical_covered_ratio=lambda df: (
            df["observed_union"] / df["theoretical"].where(df["theoretical"] != 0, 1)
        ),
        observed_both_ratio=lambda df: (
            df["both"] / df["observed_union"].where(df["observed_union"] != 0, 1)
        ),
    )
)
display(summary)

coverage.to_csv(
    CACHE_DIR / f"coverage_occupancy={OCCUPANCY_MODE}_max_atoms={MAX_ATOMS}.csv",
    index=False,
)
coverage_by_crystal_system.to_csv(
    CACHE_DIR
    / (
        f"coverage_by_crystal_system_occupancy={OCCUPANCY_MODE}"
        f"_max_atoms={MAX_ATOMS}.csv"
    ),
    index=False,
)
summary.to_csv(
    CACHE_DIR / f"summary_occupancy={OCCUPANCY_MODE}_max_atoms={MAX_ATOMS}.csv"
)
print("Wrote observed multiset, coverage, and summary CSV files to", CACHE_DIR)